In [1]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias

import circuitsvis as cv
import einops
import pandas as pd
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, display
from jaxtyping import Float, Int
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])



In [2]:
metadata_rows = [
    [data.model, data.release, data.repo_id, len(data.saes_map)] for data in get_pretrained_saes_directory().values()
]

# Print all SAE releases, sorted by base model
print(
    tabulate(
        sorted(metadata_rows, key=lambda x: x[0]),
        headers=["model", "release", "repo_id", "n_saes"],
        tablefmt="simple_outline",
    )
)


┌──────────────────────────────────────────┬─────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┬──────────┐
│ model                                    │ release                                             │ repo_id                                                   │   n_saes │
├──────────────────────────────────────────┼─────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────┼──────────┤
│ Qwen/Qwen2.5-7B-Instruct                 │ qwen2.5-7b-instruct-andyrdt                         │ andyrdt/saes-qwen2.5-7b-instruct                          │        7 │
│ deepseek-ai/DeepSeek-R1-Distill-Llama-8B │ llama_scope_r1_distill                              │ fnlp/Llama-Scope-R1-Distill                               │       96 │
│ deepseek-ai/DeepSeek-R1-Distill-Llama-8B │ deepseek-r1-distill-llama-8b-qresearch              │ qresearch/DeepSeek-R1-Distill-Llama-8B-SAE-l19     

In [3]:
def format_value(value):
    return "{{{0!r}: {1!r}, ...}}".format(*next(iter(value.items()))) if isinstance(value, dict) else repr(value)


release = get_pretrained_saes_directory()["gpt2-small-attn-out-v5-128k"]

print(
    tabulate(
        [[k, v] for k, v in release.__dict__.items()],
        headers=["Field", "Value"],
        tablefmt="simple_outline",
    )
)

┌────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ Field                  │ Value                                                                                                                                                                                                                                                                                  

In [4]:
data = [[id, path, release.neuronpedia_id[id]] for id, path in release.saes_map.items()]

print(
    tabulate(
        data,
        headers=["SAE id", "SAE path (HuggingFace)", "Neuronpedia ID"],
        tablefmt="simple_outline",
    )
)

┌─────────────────────────┬──────────────────────────┬────────────────────────────┐
│ SAE id                  │ SAE path (HuggingFace)   │ Neuronpedia ID             │
├─────────────────────────┼──────────────────────────┼────────────────────────────┤
│ blocks.0.hook_attn_out  │ v5_128k_layer_0          │ gpt2-small/0-att_128k-oai  │
│ blocks.1.hook_attn_out  │ v5_128k_layer_1          │ gpt2-small/1-att_128k-oai  │
│ blocks.2.hook_attn_out  │ v5_128k_layer_2          │ gpt2-small/2-att_128k-oai  │
│ blocks.3.hook_attn_out  │ v5_128k_layer_3          │ gpt2-small/3-att_128k-oai  │
│ blocks.4.hook_attn_out  │ v5_128k_layer_4          │ gpt2-small/4-att_128k-oai  │
│ blocks.5.hook_attn_out  │ v5_128k_layer_5          │ gpt2-small/5-att_128k-oai  │
│ blocks.6.hook_attn_out  │ v5_128k_layer_6          │ gpt2-small/6-att_128k-oai  │
│ blocks.7.hook_attn_out  │ v5_128k_layer_7          │ gpt2-small/7-att_128k-oai  │
│ blocks.8.hook_attn_out  │ v5_128k_layer_8          │ gpt2-small/8-att_128k

In [5]:
t.set_grad_enabled(False)

gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


/home/figini/Documents/projects/arena/.venv/lib/python3.13/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [6]:
print(
    tabulate(
        list(gpt2_sae.cfg.__dict__.items()) + list(gpt2_sae.cfg.metadata.items()),
        headers=["name", "value"],
        tablefmt="simple_outline",
    )
)


┌──────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ name                         │ value                                                                                                                                                                                                                                                                                                                                                                      │
├──────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [7]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=800,
    height=600,
):
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))


rand_idx = random.randint(0, gpt2_sae.cfg.d_sae)
# display_dashboard(latent_idx=rand_idx)


In [8]:
prompt = "Mitigating the risk of extinction from AI should be a global"
answer = " priority"

# First see how the model does without SAEs
test_prompt(prompt, answer, gpt2)

# Test our prompt, to see what the model says
with gpt2.saes(saes=[gpt2_sae]):
    test_prompt(prompt, answer, gpt2)

# Same thing, done in a different way
gpt2.add_sae(gpt2_sae)
test_prompt(prompt, answer, gpt2)
gpt2.reset_saes()  # Remember to always do this!

# Using `run_with_saes` method in place of standard forward pass
logits = gpt2(prompt, return_type="logits")
logits_sae = gpt2.run_with_saes(prompt, saes=[gpt2_sae], return_type="logits")
answer_token_id = gpt2.to_single_token(answer)

# Getting model's prediction
top_prob, token_id_prediction = logits[0, -1].softmax(-1).max(-1)
top_prob_sae, token_id_prediction_sae = logits_sae[0, -1].softmax(-1).max(-1)

print(f"""Standard model:
    top prediction = {gpt2.to_string(token_id_prediction)!r}
    prob = {top_prob.item():.2%}
SAE reconstruction:
    top prediction = {gpt2.to_string(token_id_prediction_sae)!r}
    prob = {top_prob_sae.item():.2%}
""")


Tokenized prompt: ['<|endoftext|>', 'Mit', 'igating', ' the', ' risk', ' of', ' extinction', ' from', ' AI', ' should', ' be', ' a', ' global']
Tokenized answer: [' priority']


Performance on answer token:
Rank: 0        Logit: 19.46 Prob: 52.99% Token: | priority|

Top 0th token. Logit: 19.46 Prob: 52.99% Token: | priority|
Top 1th token. Logit: 17.44 Prob:  7.02% Token: | effort|
Top 2th token. Logit: 16.94 Prob:  4.26% Token: | issue|
Top 3th token. Logit: 16.63 Prob:  3.14% Token: | challenge|
Top 4th token. Logit: 16.37 Prob:  2.42% Token: | goal|
Top 5th token. Logit: 16.06 Prob:  1.78% Token: | concern|
Top 6th token. Logit: 15.88 Prob:  1.47% Token: | focus|
Top 7th token. Logit: 15.61 Prob:  1.13% Token: | approach|
Top 8th token. Logit: 15.53 Prob:  1.04% Token: | policy|
Top 9th token. Logit: 15.42 Prob:  0.93% Token: | initiative|


Ranks of the answer tokens: [(' priority', 0)]

Tokenized prompt: ['<|endoftext|>', 'Mit', 'igating', ' the', ' risk', ' of', ' extinction', ' from', ' AI', ' should', ' be', ' a', ' global']
Tokenized answer: [' priority']


...

Tokenized prompt: ['<|endoftext|>', 'Mit', 'igating', ' the', ' risk', ' of', ' extinction', ' from', ' AI', ' should', ' be', ' a', ' global']
Tokenized answer: [' priority']


Performance on answer token:
Rank: 0        Logit: 18.19 Prob: 39.84% Token: | priority|

Top 0th token. Logit: 18.19 Prob: 39.84% Token: | priority|
Top 1th token. Logit: 16.51 Prob:  7.36% Token: | issue|
Top 2th token. Logit: 16.48 Prob:  7.20% Token: | concern|
Top 3th token. Logit: 15.94 Prob:  4.19% Token: | challenge|
Top 4th token. Logit: 15.30 Prob:  2.21% Token: | goal|
Top 5th token. Logit: 15.12 Prob:  1.85% Token: | responsibility|
Top 6th token. Logit: 15.04 Prob:  1.69% Token: | problem|
Top 7th token. Logit: 14.98 Prob:  1.60% Token: | effort|
Top 8th token. Logit: 14.73 Prob:  1.24% Token: | policy|
Top 9th token. Logit: 14.66 Prob:  1.16% Token: | imperative|


Ranks of the answer tokens: [(' priority', 0)]

Standard model:
    top prediction = ' priority'
    prob = 52.99%
SAE reconstruction:
    top prediction = ' priority'
    prob = 39.84%



In [9]:
_, cache = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

for name, param in cache.items():
    if "hook_sae" in name:
        print(f"{name:<43}: {tuple(param.shape)}")


blocks.7.hook_resid_pre.hook_sae_input     : (1, 13, 768)
blocks.7.hook_resid_pre.hook_sae_acts_pre  : (1, 13, 24576)
blocks.7.hook_resid_pre.hook_sae_acts_post : (1, 13, 24576)
blocks.7.hook_resid_pre.hook_sae_recons    : (1, 13, 768)
blocks.7.hook_resid_pre.hook_sae_output    : (1, 13, 768)


In [10]:
# Get top activations on final token
_, cache = gpt2.run_with_cache_with_saes(
    prompt,
    saes=[gpt2_sae],
    stop_at_layer=_get_hook_layer(gpt2_sae) + 1,
)
sae_acts_post = cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, :]

# Plot line chart of latent activations
px.line(
    sae_acts_post.cpu().numpy(),
    title=f"Latent activations at the final token position ({sae_acts_post.nonzero().numel()} alive)",
    labels={"index": "Latent", "value": "Activation"},
    width=1000,
).update_layout(showlegend=False).show()

# Print the top 5 latents, and inspect their dashboards
for act, ind in zip(*sae_acts_post.topk(3)):
    print(f"Latent {ind} had activation {act:.2f}")
    display_dashboard(latent_idx=ind)


Latent 17676 had activation 48.18
https://neuronpedia.org/gpt2-small/7-res-jb/17676?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


Latent 17053 had activation 10.77
https://neuronpedia.org/gpt2-small/7-res-jb/17053?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


Latent 792 had activation 8.90
https://neuronpedia.org/gpt2-small/7-res-jb/792?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


In [11]:
cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, 17676]

tensor(48.1769)

In [12]:
logits_no_saes, cache_no_saes = gpt2.run_with_cache(prompt)

gpt2_sae.use_error_term = False
logits_with_sae_recon, cache_with_sae_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

gpt2_sae.use_error_term = True
logits_without_sae_recon, cache_without_sae_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

# Both SAE caches contain the hook values
assert f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post" in cache_with_sae_recon
assert f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post" in cache_without_sae_recon

# But final output will be different, because we don't use SAE reconstructions when use_error_term
t.testing.assert_close(logits_no_saes, logits_without_sae_recon)
logit_diff_from_sae = (logits_no_saes - logits_with_sae_recon).abs().mean()
print(f"Average logit diff from using SAE reconstruction: {logit_diff_from_sae:.4f}")


Average logit diff from using SAE reconstruction: 0.4117


In [13]:
print(gpt2_sae.cfg.metadata.dataset_path)
gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_sae,
    dataset="NeelNanda/pile-10k",
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=str(device),
)

# Example of how you can use this:
tokens = gpt2_act_store.get_batch_tokens()
assert tokens.shape == (gpt2_act_store.store_batch_size_prompts, gpt2_act_store.context_size)


Skylion007/openwebtext


/home/figini/Documents/projects/arena/.venv/lib/python3.13/site-packages/sae_lens/training/activations_store.py:357: UserWarning: Dataset is not tokenized. Pre-tokenizing will improve performance and allows for more control over special tokens. See https://decoderesearch.github.io/SAELens/training_saes/#pretokenizing-datasets for more info.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (3180 > 1024). Running this sequence through the model will result in indexing errors


# Dashboard self-made

## activation histgram
number of tokens that activates a certain latent of the SAE

In [23]:

def show_activation_histogram(
  model: HookedSAETransformer,
  sae: SAE,
  act_store: ActivationsStore,
  latent_idx: int,
  total_batches: int = 10,
):
  """
  Displays the activation histogram for a particular latent, computed across `total_batches`
  batches from `act_store`.
  """
  sae_acts_post_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"
  all_positive_acts = []

  for i in tqdm(range(total_batches), desc="Computing activations for histogram"):
    tokens = act_store.get_batch_tokens()
    _, cache = model.run_with_cache_with_saes(
      tokens,
      saes=[sae],
      stop_at_layer=_get_hook_layer(sae) + 1,
      names_filter=[sae_acts_post_hook_name],
    )
    acts = cache[sae_acts_post_hook_name][..., latent_idx]
    all_positive_acts.extend(acts[acts > 0].cpu().tolist())

    frac_active = len(all_positive_acts) / (total_batches * act_store.store_batch_size_prompts * act_store.context_size)

  px.histogram(
    all_positive_acts,
    nbins=50,
    title=f"ACTIVATIONS DENSITY {frac_active:.3%}",
    labels={"value": "Activation"},
    width=800,
    template="ggplot2",
    color_discrete_sequence=["darkorange"],
  ).update_layout(bargap=0.02, showlegend=False).show()

show_activation_histogram(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9)

Computing activations for histogram:   0%|          | 0/10 [00:00<?, ?it/s]

## max activating samples
the prompts that show the highest level of activation from a latent

In [67]:
def topk_indices(x: Float[Tensor, "batch seq"], k: int, buffer=0) -> t.Tensor:
  print(x.shape)
  if buffer > 0: 
    x = x[:, buffer:-buffer]
  print(x.shape)
  flat_idx = x.flatten().topk(k, largest=True).indices
  rows, cols = t.unravel_index(flat_idx, x.shape)
  cols += buffer
  return t.stack((rows, cols), dim=1)

x = t.arange(40, device=device).reshape((2, 20))
x[0, 10] += 50  # 2nd highest value
x[0, 11] += 100  # highest value
x[1, 1] += 150  # not inside buffer (it's less than 3 from the start of the sequence)
top_indices = topk_indices(x, k=2, buffer=3)
print(top_indices)
assert top_indices.tolist() == [[0, 11], [0, 10]]

torch.Size([2, 20])
torch.Size([2, 14])
tensor([[ 0, 11],
        [ 0, 10]])


In [80]:
def index_with_buffer(
    x: Float[Tensor, "batch seq"], indices: Int[Tensor, "k 2"], buffer: int | None = None
) -> Float[Tensor, " k *buffer_x2_plus1"]:
    """
    Indexes into `x` with `indices` (which should have come from the `get_k_largest_indices`
    function), and takes a +-buffer range around each indexed element. If `indices` are less than
    `buffer` away from the start of a sequence then we just take the first `2*buffer+1` elems (same
    for at the end of a sequence).

    If `buffer` is None, then we don't add any buffer and just return the elements at the given indices.
    """
    rows, cols = indices.unbind(dim=-1)
    if buffer is not None:
        rows = einops.repeat(rows, "k -> k buffer", buffer=buffer * 2 + 1)
        cols[cols < buffer] = buffer
        cols[cols > x.size(1) - buffer - 1] = x.size(1) - buffer - 1
        cols = einops.repeat(cols, "k -> k buffer", buffer=buffer * 2 + 1) + t.arange(
            -buffer, buffer + 1, device=cols.device
        )
    return x[rows, cols]


x_top_values_with_context = index_with_buffer(x, top_indices, buffer=3)
assert x_top_values_with_context[0].tolist() == [
    8,
    9,
    10 + 50,
    11 + 100,
    12,
    13,
    14,
]  # highest value in the middle
assert x_top_values_with_context[1].tolist() == [
    7,
    8,
    9,
    10 + 50,
    11 + 100,
    12,
    13,
]  # 2nd highest value in the middle

In [109]:
def display_top_seqs(data: list[tuple[float, list[str], int]]):
    """
    Given a list of (activation: float, str_toks: list[str], seq_pos: int), displays a table of
    these sequences, with the relevant token highlighted.

    We also turn newlines into "\\n", and remove unknown tokens � (usually weird quotation marks)
    for readability.
    """
    table = Table("Act", "Sequence", title="Max Activating Examples", show_lines=True)
    for act, str_toks, seq_pos in data:
        formatted_seq = (
            "".join([f"[b u green]{str_tok}[/]" if i == seq_pos else str_tok for i, str_tok in enumerate(str_toks)])
            .replace("�", "")
            .replace("\n", "↵")
        )
        table.add_row(f"{act:.3f}", repr(formatted_seq))
    rprint(table)


example_data = [
    (0.5, [" one", " two", " three"], 0),
    (1.5, [" one", " two", " three"], 1),
    (2.5, [" one", " two", " three"], 2),
]
display_top_seqs(example_data)

  Max Activating Examples   
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Act   ┃ Sequence         ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 0.500 │ ' one two three' │
├───────┼──────────────────┤
│ 1.500 │ ' one two three' │
├───────┼──────────────────┤
│ 2.500 │ ' one two three' │
└───────┴──────────────────┘

In [120]:
def fetch_max_activating_examples(
  model: HookedSAETransformer,
  sae: SAE,
  act_store: ActivationsStore,
  latent_idx: int,
  total_batches: int = 10,
  k: int = 10,
  buffer: int = 10,
) -> list[tuple[float, list[str], int]]:
  """
  Returns the max activating examples across a number of batches from the activations store.
  """
  sae_acts_post_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"

  data = []
  for _ in tqdm(range(total_batches), desc="Computing activations for max activating examples"):
    tokens = act_store.get_batch_tokens()
    _, cache = model.run_with_cache_with_saes(
      tokens,
      saes=[sae],
      stop_at_layer=_get_hook_layer(sae) + 1,
      names_filter=[sae_acts_post_hook_name],
    )
    acts = cache[sae_acts_post_hook_name][..., latent_idx]
    
    top_k_idx = topk_indices(acts, k=k, buffer=buffer)
    top_k_seq = index_with_buffer(tokens, top_k_idx, buffer=buffer).to(t.int)
    str_toks = [model.to_str_tokens(toks) for toks in top_k_seq]
    top_acts = index_with_buffer(acts, top_k_idx).tolist()

    data.extend(list(zip(top_acts, str_toks, [buffer] * len(str_toks))))
    data = sorted(data, key=lambda x: x[0], reverse=True)[:k]

  return data

# Fetch & display the results
buffer = 10
data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9, buffer=buffer, k=5)
display_top_seqs(data)

# Test one of the results, to see if it matches the expected output
first_seq_str_tokens = data[0][1]
assert first_seq_str_tokens[buffer] == " new"


Computing activations for max activating examples:   0%|          | 0/10 [00:00<?, ?it/s]

torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])
torch.Size([16, 128])
torch.Size([16, 108])


                                             Max Activating Examples                                              
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Act    ┃ Sequence                                                                                              ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 23.237 │ ' an extraordinary feat of marine archaeology. Now the new £35 million, boat-shaped Mary Rose Museum' │
├────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 6.813  │ " sample succulent lobster and revel in Wight's new-found driftwood chic. Book.↵↵"                    │
├────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 6.413  │ ' trails.↵↵### CAMPING IN THE NEW FOREST↵↵The New Forest is a haven'                                  │
├────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 5.029  │ '/17).↵↵Cycling↵↵The New Forest makes for superb cycling country, with 100 miles'                     │
├────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 4.920  │ 'AMPING IN THE NEW FOREST↵↵The New Forest is a haven for campers. The Forestry'                       │
└────────┴───────────────────────────────────────────────────────────────────────────────────────────────────────┘

In [141]:
def show_top_logits(
  model: HookedSAETransformer,
  sae: SAE,
  latent_idx: int,
  k: int = 10,
) -> None:
  """
  Displays the top & bottom logits for a particular latent.
  """
  logits = sae.W_dec[latent_idx] @ gpt2.W_U
  pos_logits, pos_tok_id = logits.topk(k)
  neg_logits, neg_tok_id = logits.topk(k, largest=False)
  pos_tok = model.to_str_tokens(pos_tok_id)
  neg_tok = model.to_str_tokens(neg_tok_id)

  print(
    tabulate(
      zip(map(repr, neg_tok), neg_logits, map(repr, pos_tok), pos_logits),
      headers=["Bottom tokens", "Value", "Top tokens", "Value"],
      tablefmt="simple_outline",
      stralign="right",
      numalign="left",
      floatfmt="+.3f",
    )
  )


show_top_logits(gpt2, gpt2_sae, latent_idx=9)

┌─────────────────┬─────────┬─────────────────┬─────────┐
│   Bottom tokens │ Value   │      Top tokens │ Value   │
├─────────────────┼─────────┼─────────────────┼─────────┤
│           'Zip' │ -0.774  │          'bies' │ +1.327  │
│       'acebook' │ -0.761  │           'bie' │ +1.297  │
│           'lua' │ -0.737  │     ' arrivals' │ +1.218  │
│        'ashtra' │ -0.728  │    ' additions' │ +1.018  │
│       'ONSORED' │ -0.708  │      ' edition' │ +0.994  │
│           'OGR' │ -0.705  │   ' millennium' │ +0.966  │
│      'umenthal' │ -0.703  │   ' generation' │ +0.962  │
│        'ecause' │ -0.697  │     ' entrants' │ +0.923  │
│          'icio' │ -0.692  │        ' hires' │ +0.919  │
│          'cius' │ -0.692  │ ' developments' │ +0.881  │
└─────────────────┴─────────┴─────────────────┴─────────┘
